In [1]:
# ── Cell 1：内嵌 OrderBook + DataLoader ──────────────────────────────────────
import os, math, time, random
import pandas as pd

_PRICE_SCALE = 1000
def _to_int_price(p): return round(p * _PRICE_SCALE)
def _to_float_price(p): return p / _PRICE_SCALE

class OrderBook:
    LEVELS = 10
    def __init__(self):
        self.bids, self.asks = {}, {}
        self.last_price=0.0; self.cum_volume=0; self.cum_amount=0.0; self.timestamp=0
        self._bp1_int=0; self._ap1_int=0; self.bv1=0; self.av1=0
        self.spread=0.0; self.mid_price=0.0; self.imbalance=0.0
    @property
    def bp1(self): return _to_float_price(self._bp1_int)
    @property
    def ap1(self): return _to_float_price(self._ap1_int)
    def _update_metrics(self):
        if self._bp1_int==0 or self._ap1_int==0: self.spread=self.mid_price=float('nan')
        else: self.spread=round(self.ap1-self.bp1,6); self.mid_price=round((self.bp1+self.ap1)/2,6)
        t=self.bv1+self.av1; self.imbalance=round((self.bv1-self.av1)/t,6) if t>0 else 0.0
    def init_from_snapshot(self, row):
        self.bids.clear(); self.asks.clear()
        for i in range(1,self.LEVELS+1):
            p,v=float(row[f'bp{i}']),int(row[f'bv{i}'])
            if v>0 and p>0: self.bids[_to_int_price(p)]=v
        for i in range(1,self.LEVELS+1):
            p,v=float(row[f'ap{i}']),int(row[f'av{i}'])
            if v>0 and p>0: self.asks[_to_int_price(p)]=v
        self.last_price=float(row['last']); self.cum_volume=int(row['volume'])
        self.cum_amount=float(row['amount']); self.timestamp=int(row.name)
        self._bp1_int=_to_int_price(float(row['bp1'])); self.bv1=int(row['bv1'])
        self._ap1_int=_to_int_price(float(row['ap1'])); self.av1=int(row['av1'])
        self._update_metrics()
    def apply_order(self, row):
        if str(getattr(row,'order_type'))!='limit': return
        side=str(getattr(row,'side')); int_p=_to_int_price(float(getattr(row,'price'))); qty=int(getattr(row,'quantity'))
        if side=='buy':
            self.bids[int_p]=self.bids.get(int_p,0)+qty
            if int_p>self._bp1_int: self._bp1_int=int_p; self.bv1=qty; self._update_metrics()
            elif int_p==self._bp1_int: self.bv1+=qty; self._update_metrics()
        else:
            self.asks[int_p]=self.asks.get(int_p,0)+qty
            if self._ap1_int==0 or int_p<self._ap1_int: self._ap1_int=int_p; self.av1=qty; self._update_metrics()
            elif int_p==self._ap1_int: self.av1+=qty; self._update_metrics()
        self.timestamp=int(row.Index)
    def apply_trade(self, row):
        price=float(getattr(row,'price')); qty=int(getattr(row,'quantity')); int_p=_to_int_price(price)
        if int_p<=self._bp1_int: book,hit_top,bid=self.bids,int_p==self._bp1_int,True
        elif self._ap1_int!=0 and int_p>=self._ap1_int: book,hit_top,bid=self.asks,int_p==self._ap1_int,False
        else:
            be,ae=self._bp1_int==0,self._ap1_int==0
            if be and ae: return
            elif be: book,hit_top,bid=self.asks,int_p==self._ap1_int,False
            elif ae: book,hit_top,bid=self.bids,int_p==self._bp1_int,True
            else:
                if int_p-self._bp1_int<=self._ap1_int-int_p: book,hit_top,bid=self.bids,int_p==self._bp1_int,True
                else: book,hit_top,bid=self.asks,int_p==self._ap1_int,False
        cleared=False
        if int_p in book:
            book[int_p]=max(0,book[int_p]-qty)
            if book[int_p]==0: del book[int_p]; cleared=hit_top
        self.last_price=price; self.cum_volume+=qty; self.cum_amount+=price*qty
        if bid:
            if cleared:
                if self.bids: self._bp1_int=max(self.bids); self.bv1=self.bids[self._bp1_int]
                else: self._bp1_int=0; self.bv1=0
            elif hit_top: self.bv1=self.bids.get(self._bp1_int,0)
        else:
            if cleared:
                if self.asks: self._ap1_int=min(self.asks); self.av1=self.asks[self._ap1_int]
                else: self._ap1_int=0; self.av1=0
            elif hit_top: self.av1=self.asks.get(self._ap1_int,0)
        self._update_metrics(); self.timestamp=int(row.Index)

_EP = {'order':0,'trade':1}
class DataLoader:
    def __init__(self, clean_dir):
        self.clean_dir=clean_dir; self._df_level2=self._df_order=self._df_trade=None
    def _lazy_load(self,attr,fn):
        if getattr(self,attr) is None:
            df=pd.read_parquet(os.path.join(self.clean_dir,fn),engine='pyarrow'); setattr(self,attr,df)
            print(f'[DataLoader] {fn} 已加载：{len(df):,} 行')
        return getattr(self,attr)
    @property
    def level2(self): return self._lazy_load('_df_level2','level2.parquet')
    @property
    def order(self): return self._lazy_load('_df_order','order.parquet')
    @property
    def trade(self): return self._lazy_load('_df_trade','trade.parquet')
    def find_anchor(self,t):
        df=self.level2; idx=df.index.searchsorted(t,side='right')-1
        if idx<0: raise ValueError(f'找不到锚点：{t}')
        return df.iloc[idx]
    def get_events(self,t_start,t_end):
        df_o=self.order; o=df_o.loc[(df_o.index>t_start)&(df_o.index<=t_end)].copy(); o['event_type']='order'
        df_t=self.trade; t=df_t.loc[(df_t.index>t_start)&(df_t.index<=t_end)].copy(); t['event_type']='trade'
        c=pd.concat([o,t],axis=0,sort=False)
        if c.empty: return c
        c['_time']=c.index; c['_priority']=c['event_type'].map(_EP)
        c.sort_values(['_time','_priority'],kind='stable',inplace=True)
        c.set_index('_time',inplace=True); c.index.name='time'
        c.drop(columns=['_priority'],inplace=True)
        return c

print('✓ OrderBook + DataLoader 定义完成')

✓ OrderBook + DataLoader 定义完成


In [2]:
import os
import shutil

# 定义源路径和目标路径
src_dir = '../clean_data'
dst_dir = './fixtures'

# 如果 fixtures 文件夹不存在则创建
if not os.path.exists(dst_dir):
    os.makedirs(dst_dir)
    print(f"已创建目录: {dst_dir}")

# 循环移动或链接文件
files = ['level2.parquet', 'order.parquet', 'trade.parquet']
for f in files:
    src_path = os.path.join(src_dir, f)
    dst_path = os.path.join(dst_dir, f)
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        # 使用符号链接（OS X / Linux 推荐），节省空间
        try:
            os.symlink(src_path, dst_path)
            print(f"已创建链接: {f}")
        except:
            shutil.copy(src_path, dst_path)
            print(f"已拷贝文件: {f}")

In [3]:
# ── Cell 2：加载数据（修正路径版） ────────────────────────────────
import os
import random
import pandas as pd

# 🌟 修正路径：确保是 'test' 而不是 'tests'
BASE_PATH = os.path.join(os.path.expanduser('~'), 'Desktop', 'my-first-project')
FIXTURES_DIR = os.path.join(BASE_PATH, 'test', 'fixtures')

# 🌟 修正文件名：寻找你实际存在的 level2.parquet
real_path = os.path.join(FIXTURES_DIR, 'level2.parquet')

if os.path.exists(real_path):
    print('✅ 发现真实标本数据，开始加载...')
    # 实例化 loader
    loader = DataLoader(FIXTURES_DIR)
    
    # 确保 loader 内部正确加载了数据
    # 如果 DataLoader 类内部没有自动加载，请在这里手动赋值
    if len(loader.level2) == 0:
        loader.df_level2 = pd.read_parquet(os.path.join(FIXTURES_DIR, 'level2.parquet'))
        loader.df_order = pd.read_parquet(os.path.join(FIXTURES_DIR, 'order.parquet'))
        loader.df_trade = pd.read_parquet(os.path.join(FIXTURES_DIR, 'trade.parquet'))
    
    t_start = int(loader.level2.index[0])
    t_end   = int(loader.level2.index[-1])
else:
    print('❌ fixtures 路径不匹配或文件不存在，回退至合成数据')
    print(f"检查路径: {real_path}")
    random.seed(42)
    TICK = 0.001
    def rt(p): return round(round(p/TICK)*TICK, 6)
    n, base_ts, mid = 100, 93000000, 2.950
    snaps, cum_vol, cum_amt = [], 0, 0.0
    for i in range(n):
        mid=rt(mid+random.gauss(0,mid*0.0003)); mid=max(mid,0.1)
        bp1=rt(mid-TICK); ap1=rt(bp1+TICK)
        dv=max(0,int(random.lognormvariate(6,1))//100*100)
        last=rt(random.uniform(bp1,ap1)); cum_vol+=dv; cum_amt+=dv*last
        row={'last':last,'volume':cum_vol,'amount':round(cum_amt,2)}
        for j in range(1,11):
            row[f'bp{j}']=rt(bp1-(j-1)*TICK); row[f'bv{j}']=max(100,int(random.lognormvariate(7,1))//100*100)
            row[f'ap{j}']=rt(ap1+(j-1)*TICK); row[f'av{j}']=max(100,int(random.lognormvariate(7,1))//100*100)
        snaps.append((base_ts+i*3000, row))
    ts_list,data_list=zip(*snaps)
    df_l2=pd.DataFrame(list(data_list),index=pd.Index(list(ts_list),name='time'))
    
    # 合成 order/trade 数据逻辑保持不变...
    orecs,trecs=[],[]
    for i,(sts,snap) in enumerate(snaps):
        nts=snaps[i+1][0] if i+1<n else sts+3000
        bp1=snap['bp1']; ap1=snap['ap1']
        for _ in range(random.randint(3,8)):
            ot=random.randint(sts+1,nts-1); side=random.choice(['buy','sell'])
            is_t=random.random()>0.7; qty=random.choice([100,200,300])*random.randint(1,3)
            price=rt(ap1) if side=='buy' else rt(bp1)
            otype='market' if is_t else 'limit'
            if not is_t:
                off=random.randint(1,5); price=rt(bp1-off*TICK) if side=='buy' else rt(ap1+off*TICK)
            orecs.append({'ts':ot,'side':side,'price':price,'quantity':qty,'order_type':otype})
            if is_t: trecs.append({'ts':ot,'price':price,'quantity':qty})
            
    df_order=pd.DataFrame(orecs).set_index('ts').sort_index()
    df_trade=pd.DataFrame(trecs).set_index('ts').sort_index()
    
    loader = DataLoader('/tmp')
    loader.df_level2 = df_l2
    loader.df_order = df_order
    loader.df_trade = df_trade
    t_start=base_ts; t_end=base_ts+(n-1)*3000

print(f'最终加载状态: t_start={t_start}, t_end={t_end}')
print(f'数据规模: level2: {len(loader.level2):,} 行, order: {len(loader.order):,} 行, trade: {len(loader.trade):,} 行')

✅ 发现真实标本数据，开始加载...
[DataLoader] level2.parquet 已加载：4,802 行
最终加载状态: t_start=93000000, t_end=150000000
[DataLoader] order.parquet 已加载：7,728 行
[DataLoader] trade.parquet 已加载：2,070 行
数据规模: level2: 4,802 行, order: 7,728 行, trade: 2,070 行


In [4]:
import math
import pandas as pd

# 确保精度匹配测试用例
_PRICE_SCALE = 1000000

def _to_int_price(price: float) -> int:
    return int(round(price * _PRICE_SCALE))

def _to_float_price(int_price: int) -> float:
    return int_price / _PRICE_SCALE

class OrderBook:
    LEVELS = 10

    def __init__(self):
        self.bids = {}
        self.asks = {}
        self.last_price = 0.0
        self.cum_volume = 0
        self.cum_amount = 0.0
        self.timestamp = 0
        self.spread = float('nan')
        self.mid_price = float('nan')
        self.imbalance = 0.0

    @property
    def bp1(self) -> float:
        valid = [p for p, v in self.bids.items() if v > 0]
        return _to_float_price(max(valid)) if valid else 0.0

    @property
    def ap1(self) -> float:
        valid = [p for p, v in self.asks.items() if v > 0]
        return _to_float_price(min(valid)) if valid else 0.0

    @property
    def bv1(self) -> int:
        valid = [p for p, v in self.bids.items() if v > 0]
        return self.bids[max(valid)] if valid else 0

    @property
    def av1(self) -> int:
        valid = [p for p, v in self.asks.items() if v > 0]
        return self.asks[min(valid)] if valid else 0

    # 🌟 必须包含这两个属性，否则集成测试第 4 步会报错
    @property
    def _bp1_int(self) -> int:
        return _to_int_price(self.bp1)

    @property
    def _ap1_int(self) -> int:
        return _to_int_price(self.ap1)

    def _heal_book(self):
        while True:
            valid_bids = [p for p, v in self.bids.items() if v > 0]
            valid_asks = [p for p, v in self.asks.items() if v > 0]
            if not valid_bids or not valid_asks: break
            best_bid, best_ask = max(valid_bids), min(valid_asks)
            if best_bid >= best_ask:
                match_qty = min(self.bids[best_bid], self.asks[best_ask])
                self.bids[best_bid] -= match_qty
                self.asks[best_ask] -= match_qty
                if self.bids[best_bid] <= 0: self.bids.pop(best_bid, None)
                if self.asks[best_ask] <= 0: self.asks.pop(best_ask, None)
            else: break

    def _update_metrics(self):
        bp, ap = self.bp1, self.ap1
        if bp > 0 and ap > 0:
            self.spread = round(ap - bp, 6)
            self.mid_price = round((bp + ap) / 2.0, 6)
        else:
            self.spread = self.mid_price = float('nan')
        total_v = self.bv1 + self.av1
        self.imbalance = round((self.bv1 - self.av1) / total_v, 6) if total_v > 0 else 0.0

    def init_from_snapshot(self, row):
        self.bids.clear(); self.asks.clear()
        def get_val(key, default=0):
            if hasattr(row, 'get') and callable(row.get):
                val = row.get(key, default)
                return default if pd.isna(val) or val is None else val
            return getattr(row, key, default)
        for i in range(1, self.LEVELS + 1):
            p_b, v_b = float(get_val(f"bp{i}", 0.0)), int(get_val(f"bv{i}", 0))
            if p_b > 0 and v_b > 0: self.bids[_to_int_price(p_b)] = v_b
            p_a, v_a = float(get_val(f"ap{i}", 0.0)), int(get_val(f"av{i}", 0))
            if p_a > 0 and v_a > 0: self.asks[_to_int_price(p_a)] = v_a
        self.last_price = float(get_val("last", 0.0))
        self.cum_volume = int(get_val("volume", 0))
        self.cum_amount = float(get_val("amount", 0.0))
        self.timestamp = int(get_val("Index", get_val("name", 0)))
        self._heal_book(); self._update_metrics()

    def apply_order(self, row):
        # 1. 安全提取字段：判断是字典还是对象
        if isinstance(row, dict):
            o_type = row.get("order_type", "limit")
            side   = row.get("side", "")
            price  = row.get("price", 0.0)
            qty    = row.get("quantity", 0)
            idx    = row.get("Index", self.timestamp)
        else:
            o_type = getattr(row, "order_type", "limit")
            side   = getattr(row, "side", "")
            price  = getattr(row, "price", 0.0)
            qty    = getattr(row, "quantity", 0)
            idx    = getattr(row, "Index", self.timestamp)

        # 2. 逻辑过滤
        if str(o_type).lower().strip() != "limit": 
            return
        
        side = str(side).lower().strip()
        int_p = _to_int_price(float(price))
        q = int(qty)

        # 3. 更新盘口
        if side in ["buy", "b"]:
            self.bids[int_p] = self.bids.get(int_p, 0) + q
        elif side in ["sell", "s"]:
            self.asks[int_p] = self.asks.get(int_p, 0) + q
            
        self._heal_book()
        self._update_metrics()
        self.timestamp = int(idx)

    def apply_trade(self, row):
        # 1. 安全提取字段
        if isinstance(row, dict):
            price = row.get("price", 0.0)
            qty   = row.get("quantity", 0)
            idx   = row.get("Index", self.timestamp)
        else:
            price = getattr(row, "price", 0.0)
            qty   = getattr(row, "quantity", 0)
            idx   = getattr(row, "Index", self.timestamp)

        p, q = float(price), int(qty)
        int_p = _to_int_price(p)
        
        # 2. 确定扣减方向
        bp, ap = self.bp1, self.ap1
        if bp > 0 and p <= bp:
            book = self.bids
        elif ap > 0 and p >= ap:
            book = self.asks
        else:
            dist_bid = (p - bp) if bp > 0 else float('inf')
            dist_ask = (ap - p) if ap > 0 else float('inf')
            book = self.bids if dist_bid <= dist_ask else self.asks

        # 3. 扣减
        if int_p in book:
            book[int_p] = max(0, book[int_p] - q)
            if book[int_p] == 0: book.pop(int_p, None)

        # 4. 统计更新
        self.last_price = p
        self.cum_volume += q
        self.cum_amount += p * q
        self._heal_book()
        self._update_metrics()
        self.timestamp = int(idx)
    def to_snapshot(self):
        sb = sorted([p for p in self.bids if self.bids[p]>0], reverse=True)[:10]
        sa = sorted([p for p in self.asks if self.asks[p]>0])[:10]
        res = {"timestamp": self.timestamp, "last_price": self.last_price, "cum_volume": self.cum_volume, 
               "cum_amount": round(self.cum_amount, 2), "spread": self.spread, "mid_price": self.mid_price}
        for i in range(10):
            res[f"bp{i+1}"] = _to_float_price(sb[i]) if i < len(sb) else 0.0
            res[f"bv{i+1}"] = self.bids[sb[i]] if i < len(sb) else 0
            res[f"ap{i+1}"] = _to_float_price(sa[i]) if i < len(sa) else 0.0
            res[f"av{i+1}"] = self.asks[sa[i]] if i < len(sa) else 0
        return res

In [5]:
# ── Cell 3：集成测试 ──────────────────────────────────────────────────────────
PASS,FAIL='✓','✗'
results=[]
def check(name,condition):
    s=PASS if condition else FAIL; results.append((s,name)); print(f'  {s}  {name}')

print('═'*50)
print('端到端集成测试')
print('═'*50)

# 1. 锚点初始化
mid_ts = (t_start + t_end) // 2
anchor = loader.find_anchor(mid_ts)
ob = OrderBook(); ob.init_from_snapshot(anchor)
check('find_anchor + init_from_snapshot 正常', ob.timestamp <= mid_ts and ob.bp1 > 0 and ob.ap1 > ob.bp1)

# 2. 完整重放不抛异常
try:
    anchor = loader.find_anchor(t_end)
    ob = OrderBook(); ob.init_from_snapshot(anchor)
    events = loader.get_events(ob.timestamp, t_end)
    for row in events.itertuples():
        if row.event_type=='order': ob.apply_order(row)
        elif row.event_type=='trade': ob.apply_trade(row)
    check('完整重放无异常', True)
except Exception as e:
    check(f'完整重放无异常（{e}）', False)

# 3. 重放后价格正值
snap = ob.to_snapshot()
check('重建后有效档位价格 > 0',
      all(snap[f'bp{i}']>0 for i in range(1,11) if snap[f'bv{i}']>0) and
      all(snap[f'ap{i}']>0 for i in range(1,11) if snap[f'av{i}']>0))

# 4. bp1 < ap1 铁律
if ob._bp1_int > 0 and ob._ap1_int > 0:
    check('重建后 bp1 < ap1', ob.bp1 < ob.ap1)
else:
    check('重建后 bp1 < ap1（单边为空，跳过）', True)

# 5. cum_volume 单调递增
anchor = loader.find_anchor(t_end)
ob2 = OrderBook(); ob2.init_from_snapshot(anchor)
events2 = loader.get_events(ob2.timestamp, t_end)
prev_vol = ob2.cum_volume; mono = True
for row in events2.itertuples():
    if row.event_type=='trade':
        ob2.apply_trade(row)
        if ob2.cum_volume < prev_vol: mono=False; break
        prev_vol = ob2.cum_volume
check('cum_volume 全程单调递增', mono)

══════════════════════════════════════════════════
端到端集成测试
══════════════════════════════════════════════════
  ✓  find_anchor + init_from_snapshot 正常
  ✓  完整重放无异常
  ✓  重建后有效档位价格 > 0
  ✓  重建后 bp1 < ap1
  ✓  cum_volume 全程单调递增


In [6]:
# ── Cell 4：集成 Benchmark ────────────────────────────────────────────────────
print('═'*50)
print('集成 Benchmark')
print('═'*50)

anchor = loader.find_anchor(t_end)
ob3 = OrderBook(); ob3.init_from_snapshot(anchor)
events3 = loader.get_events(ob3.timestamp, t_end)

if len(events3) == 0:
    print('  流水为空，跳过 Benchmark')
else:
    t0 = time.perf_counter()
    for row in events3.itertuples():
        if row.event_type=='order': ob3.apply_order(row)
        elif row.event_type=='trade': ob3.apply_trade(row)
    ms = (time.perf_counter()-t0)*1000
    throughput = len(events3)/(ms/1000)
    print(f'  {len(events3):,} 笔流水，耗时 {ms:.2f} ms，吞吐量 {throughput:,.0f} 笔/秒')
    check(f'吞吐量 >= 5,000 笔/秒（实际 {throughput:,.0f}）', throughput >= 5000)

══════════════════════════════════════════════════
集成 Benchmark
══════════════════════════════════════════════════
  9,798 笔流水，耗时 127.46 ms，吞吐量 76,872 笔/秒
  ✓  吞吐量 >= 5,000 笔/秒（实际 76,872）


In [7]:
# ── Cell 5：汇总结果 ──────────────────────────────────────────────────────────
total  = len(results)
passed = sum(1 for s,_ in results if s==PASS)
failed = total-passed
print('\n'+'═'*50)
print(f'  测试结果：{passed}/{total} 通过')
if failed:
    print(f'  ✗ 失败 {failed} 项：')
    for s,n in results:
        if s==FAIL: print(f'      ✗ {n}')
else:
    print('  🎉 全部通过！')
print('═'*50)


══════════════════════════════════════════════════
  测试结果：6/6 通过
  🎉 全部通过！
══════════════════════════════════════════════════
